# 07 · Nexus RecSys — Modelado y Evaluación

**Sistema de Recomendación E-Commerce · RetailRocket Dataset**

---

Este notebook implementa el pipeline completo de modelado para Nexus RecSys:

| Paso | Contenido |
|------|-----------|
| 1 | Carga y validación de datos procesados |
| 2 | Definición del problema y reconstrucción del split temporal |
| 3 | Framework de evaluación centralizado (Precision, Recall, NDCG, MAP, Coverage, Novelty) |
| 4a | Popularity Baseline |
| 4b | SVD — Matrix Factorization (scipy) |
| 4c | NMF — Factorización No-Negativa / ALS-like (sklearn) |
| 4d | LightGBM Learning-to-Rank |
| 4e | Item-Based Collaborative Filtering (Item-CF via embeddings NMF) |
| 4f | Content-Based Filtering (features de ítems + pesos por tipo de interacción) |
| 5 | Optimización con Optuna (SVD y LightGBM) |
| 5c | Modelo Híbrido CF + CBF (búsqueda de α en validation set) |
| 6 | Tabla comparativa final con todos los modelos |
| 7 | Selección del modelo final y justificación |
| 8 | Ejemplos reales de recomendación |
| 9 | Análisis de trade-offs, limitaciones y guardado de artefactos |

> **Reproducibilidad:** `random_state=42` fijado en todos los pasos.
> **Evaluación:** Solo sobre usuarios con historial de *train* **y** *test* (warm users).


## 0 · Setup y Configuración Global

In [ ]:
import os, time, json, pickle, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import svds

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle as sk_shuffle

import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
logging.getLogger("lightgbm").setLevel(logging.ERROR)

# ── Semillas reproducibles ─────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Parámetros globales ────────────────────────────────────────────────────────
CUTOFF_DATE   = pd.Timestamp("2015-08-22", tz="UTC")
K_VALUES      = [5, 10]
N_EVAL_USERS  = 3_000     # muestra para evaluación
N_FACTORS     = 50        # factores latentes para MF
BATCH_SIZE    = 512       # batch de usuarios en predicción matricial
N_NEG_RATIO   = 3         # negativos por positivo en LightGBM
N_OPTUNA      = 20        # trials Optuna
MIN_TRAIN_ITEMS = 1       # mínimo de items en train para considerar usuario
MIN_TEST_ITEMS  = 1       # mínimo de items en test para considerar usuario

# ── Rutas ──────────────────────────────────────────────────────────────────────
HERE      = Path().resolve()
ROOT      = HERE.parent if (HERE.parent / "data").exists() else HERE
DATA_DIR  = ROOT / "data" / "processed"
ENC_DIR   = ROOT / "encoders"
DOCS_DIR  = ROOT / "docs"
DOCS_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")

print(f"Proyecto : {ROOT}")
print(f"Datos    : {DATA_DIR}")
print(f"Cutoff   : {CUTOFF_DATE.date()}")
print(f"K values : {K_VALUES}")
print(f"Factores : {N_FACTORS}")
print(f"Eval usr : {N_EVAL_USERS:,}")


## 1 · Carga y Validación de Datos

Se cargan los 5 artefactos producidos en el pipeline de Feature Engineering.
Los tipos de datos y nulos se verifican antes de cualquier transformación.


In [ ]:
# ── Carga ──────────────────────────────────────────────────────────────────────
print("Cargando artefactos de data/processed/ ...")

im  = pd.read_csv(DATA_DIR / "interaction_matrix.csv")
uf  = pd.read_csv(DATA_DIR / "user_features.csv")
itf = pd.read_csv(DATA_DIR / "item_features.csv")

with open(DATA_DIR / "train_test_split_info.json") as f:
    split_info = json.load(f)

fp  = pd.read_parquet(DATA_DIR / "cp06_features_final.parquet")

print("\n=== interaction_matrix ===")
print(f"  Shape  : {im.shape}")
print(f"  Cols   : {im.columns.tolist()}")
print(f"  Nulos  : {im.isnull().sum().sum()}")

print("\n=== user_features ===")
print(f"  Shape  : {uf.shape}")
print(f"  Nulos  : {uf.isnull().sum().sum()}")

print("\n=== item_features ===")
print(f"  Shape  : {itf.shape}")
print(f"  Nulos  : {itf.isnull().sum().sum()}")

print("\n=== train_test_split_info ===")
for k, v in split_info.items():
    print(f"  {k}: {v}")

print("\n=== cp06_features_final ===")
print(f"  Shape  : {fp.shape}")


In [ ]:
# ── Estadísticas clave del dataset ─────────────────────────────────────────────
n_users = im["visitorid"].nunique()
n_items = im["itemid"].nunique()
n_ints  = len(im)
sparsity = 1.0 - n_ints / (n_users * n_items)

print("=" * 55)
print("  ESTADÍSTICAS CLAVE DEL DATASET")
print("=" * 55)
print(f"  Usuarios únicos       : {n_users:>12,}")
print(f"  Ítems únicos          : {n_items:>12,}")
print(f"  Interacciones totales : {n_ints:>12,}")
print(f"  Sparsity              : {sparsity:.6f}  ({sparsity*100:.4f}%)")
print()

pu = im.groupby("visitorid")["itemid"].count()
pi = im.groupby("itemid")["visitorid"].count()

print("  Distribución interacciones/usuario:")
print(pu.describe().apply(lambda x: f"{x:>10.2f}").to_string())
print()
print("  Distribución interacciones/ítem:")
print(pi.describe().apply(lambda x: f"{x:>10.2f}").to_string())
print()
print("  Distribución interaction_strength:")
print(im["interaction_strength"].value_counts().head(10).to_string())


In [ ]:
# ── Visualizaciones del dataset ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1) Distribución de interacciones por usuario (log)
pu_log = pu[pu > 0]
axes[0].hist(pu_log, bins=60, color=sns.color_palette("muted")[0], edgecolor="white")
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_title("Interacciones / Usuario (log-log)")
axes[0].set_xlabel("# ítems"); axes[0].set_ylabel("# usuarios")

# 2) Distribución de interaction_strength
strength_counts = im["interaction_strength"].value_counts().sort_index().head(10)
axes[1].bar(strength_counts.index.astype(str), strength_counts.values,
            color=sns.color_palette("muted")[1], edgecolor="white")
axes[1].set_title("Distribución de interaction_strength")
axes[1].set_xlabel("Strength"); axes[1].set_ylabel("# interacciones")

# 3) Interacciones por ítem (log)
axes[2].hist(pi[pi > 0], bins=60, color=sns.color_palette("muted")[2], edgecolor="white")
axes[2].set_xscale("log"); axes[2].set_yscale("log")
axes[2].set_title("Interacciones / Ítem (log-log)")
axes[2].set_xlabel("# usuarios"); axes[2].set_ylabel("# ítems")

plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_dataset_stats.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada en docs/fig_dataset_stats.png")


## 2 · Reconstrucción del Split Temporal

El split es **temporal**: las interacciones cuya `last_interaction_ts < 2015-08-22`
forman el **train**; las que ocurrieron en o después del cutoff forman el **test**.

Para que un usuario pueda ser evaluado necesita:
- Al menos **1 ítem en train** (historial conocido por el modelo)
- Al menos **1 ítem en test** (ground truth para medir calidad)

Estos usuarios se denominan **warm evaluation users**.


In [ ]:
# ── Parsear timestamps ──────────────────────────────────────────────────────────
im["last_interaction_ts"] = pd.to_datetime(im["last_interaction_ts"], utc=True)

# ── Split temporal ──────────────────────────────────────────────────────────────
train_mask = im["last_interaction_ts"] < CUTOFF_DATE
test_mask  = ~train_mask

train_df = im[train_mask].copy()
test_df  = im[test_mask].copy()

# ── Usuarios cálidos (warm) ─────────────────────────────────────────────────────
train_users_set = set(train_df["visitorid"].unique())
test_users_set  = set(test_df["visitorid"].unique())
warm_users      = sorted(train_users_set & test_users_set)

print(f"Interacciones TRAIN : {len(train_df):>12,}  ({len(train_df)/len(im)*100:.1f}%)")
print(f"Interacciones TEST  : {len(test_df):>12,}  ({len(test_df)/len(im)*100:.1f}%)")
print(f"Usuarios con TRAIN  : {len(train_users_set):>12,}")
print(f"Usuarios con TEST   : {len(test_users_set):>12,}")
print(f"Usuarios WARM (eval): {len(warm_users):>12,}")
print()
print(f"Fracción evaluable  : {len(warm_users)/n_users*100:.2f}% del total de usuarios")

# ── Cold-start stats ─────────────────────────────────────────────────────────
cold_start_users = test_users_set - train_users_set
print(f"Usuarios cold-start : {len(cold_start_users):>12,} (sin historial en train)")


In [ ]:
# ── Generar estructuras de acceso rápido ────────────────────────────────────────
train_items_by_user = (
    train_df.groupby("visitorid")["itemid"]
    .apply(set)
    .to_dict()
)
test_items_by_user = (
    test_df.groupby("visitorid")["itemid"]
    .apply(set)
    .to_dict()
)

# ── Muestra de evaluación ────────────────────────────────────────────────────────
rng = np.random.default_rng(RANDOM_STATE)
if len(warm_users) > N_EVAL_USERS:
    eval_users = rng.choice(warm_users, size=N_EVAL_USERS, replace=False).tolist()
else:
    eval_users = warm_users

print(f"Usuarios en muestra de evaluación : {len(eval_users):,}")

# ── Índices globales (toda la matriz) ────────────────────────────────────────────
all_items_global = sorted(im["itemid"].unique())
item2idx_global  = {it: i for i, it in enumerate(all_items_global)}
n_items_global   = len(all_items_global)

print(f"Catálogo total de ítems           : {n_items_global:,}")


In [ ]:
# ── Construir matriz dispersa de entrenamiento ──────────────────────────────────
all_train_users = sorted(train_df["visitorid"].unique())
all_train_items = sorted(train_df["itemid"].unique())

user2idx = {u: i for i, u in enumerate(all_train_users)}
item2idx = {it: i for i, it in enumerate(all_train_items)}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: it for it, i in item2idx.items()}

n_train_users = len(all_train_users)
n_train_items = len(all_train_items)

rows = train_df["visitorid"].map(user2idx).values
cols = train_df["itemid"].map(item2idx).values
vals = train_df["interaction_strength"].values.astype(np.float32)

# Matriz CSR usuario × ítem
train_matrix = sp.csr_matrix(
    (vals, (rows, cols)),
    shape=(n_train_users, n_train_items),
    dtype=np.float32
)

# Popularidad de ítems en train (usada para Novelty y Popularity Baseline)
item_pop_train = (
    train_df.groupby("itemid")["visitorid"]
    .count()
    .rename("pop")
)

print(f"Matriz de entrenamiento shape : {train_matrix.shape}")
print(f"Non-zeros                     : {train_matrix.nnz:,}")
print(f"Sparsity train matrix         : {1 - train_matrix.nnz / (n_train_users * n_train_items):.6f}")


## 3 · Framework de Evaluación Centralizado

Todas las métricas están implementadas en la función `evaluate_model()`, reutilizable 
para cada modelo. Las métricas estándar de sistemas de recomendación son:

| Métrica | Descripción |
|---------|-------------|
| **Precision@K** | Fracción de los K recomendados que son relevantes |
| **Recall@K** | Fracción de los ítems relevantes recuperados en los top-K |
| **NDCG@K** | Calidad del ranking, pondera posición del ítem relevante |
| **MAP@K** | Precisión promedio ponderada por posición |
| **Coverage** | % del catálogo total cubierto por las recomendaciones |
| **Novelty** | Media de la auto-información (−log₂ popularidad) de los ítems recomendados |

### Justificación de métricas

Para e-commerce con feedback implícito (la relevancia es binaria: interactuó o no),
NDCG y MAP son más informativas que RMSE/MAE porque capturan la *calidad del ranking*.
Coverage y Novelty son fundamentales para evitar el "filter bubble" de popularidad.


In [ ]:
# ── Funciones de métrica ────────────────────────────────────────────────────────

def precision_at_k(recs, relevant_set, k):
    return len(set(recs[:k]) & relevant_set) / k if k > 0 else 0.0

def recall_at_k(recs, relevant_set, k):
    if not relevant_set:
        return 0.0
    return len(set(recs[:k]) & relevant_set) / len(relevant_set)

def dcg_at_k(recs, relevant_set, k):
    return sum(
        1.0 / np.log2(i + 2)
        for i, item in enumerate(recs[:k])
        if item in relevant_set
    )

def ndcg_at_k(recs, relevant_set, k):
    ideal = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
    return dcg_at_k(recs, relevant_set, k) / ideal if ideal > 0 else 0.0

def ap_at_k(recs, relevant_set, k):
    if not relevant_set:
        return 0.0
    score, hits = 0.0, 0
    for i, item in enumerate(recs[:k]):
        if item in relevant_set:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant_set), k)

def novelty_score(recs, item_pop_dict, n_train_total):
    '''Auto-información media: mayor valor = ítems menos populares.'''
    scores = [
        -np.log2(item_pop_dict.get(it, 1) / n_train_total + 1e-10)
        for it in recs
    ]
    return np.mean(scores) if scores else 0.0

print("Funciones de métricas definidas (precision, recall, ndcg, ap, novelty)")


In [ ]:
# ── Función centralizada de evaluación ─────────────────────────────────────────

def evaluate_model(
    get_recs_fn,
    eval_user_ids,
    test_items_by_user,
    train_items_by_user,
    item_pop_dict,
    n_train_total,
    catalog_size,
    k_values=[5, 10]
):
    """
    Evalúa un modelo de recomendación sobre el conjunto de test.

    Parámetros
    ----------
    get_recs_fn : callable(user_id, n) -> list[item_id]
        Función que genera top-n recomendaciones para un usuario.
    eval_user_ids : list
        Lista de IDs de usuarios a evaluar.
    test_items_by_user : dict
        {user_id: set(item_ids)} — ground truth test.
    train_items_by_user : dict
        {user_id: set(item_ids)} — para excluir de recomendaciones.
    item_pop_dict : dict
        {item_id: n_interactions_train}
    n_train_total : int
        Total de interacciones en train.
    catalog_size : int
        Total de ítems en el catálogo.
    k_values : list
        Valores de K para calcular métricas.

    Retorna
    -------
    dict : métricas agregadas
    """
    accum = {k: {"p": [], "r": [], "ndcg": [], "map": []} for k in k_values}
    all_recommended = set()
    n_evaluated = 0

    for uid in eval_user_ids:
        test_items = test_items_by_user.get(uid, set())
        if not test_items:
            continue

        max_k = max(k_values)
        try:
            recs = get_recs_fn(uid, max_k)
        except Exception:
            continue

        all_recommended.update(recs)
        n_evaluated += 1

        for k in k_values:
            accum[k]["p"].append(precision_at_k(recs, test_items, k))
            accum[k]["r"].append(recall_at_k(recs, test_items, k))
            accum[k]["ndcg"].append(ndcg_at_k(recs, test_items, k))
            accum[k]["map"].append(ap_at_k(recs, test_items, k))

    metrics = {"n_evaluated": n_evaluated}
    for k in k_values:
        if accum[k]["p"]:
            metrics[f"Precision@{k}"] = float(np.mean(accum[k]["p"]))
            metrics[f"Recall@{k}"]    = float(np.mean(accum[k]["r"]))
            metrics[f"NDCG@{k}"]      = float(np.mean(accum[k]["ndcg"]))
            metrics[f"MAP@{k}"]       = float(np.mean(accum[k]["map"]))

    metrics["Coverage"] = len(all_recommended) / catalog_size if catalog_size > 0 else 0.0

    # Novelty promedio sobre todos los ítems recomendados
    metrics["Novelty"] = novelty_score(list(all_recommended), item_pop_dict, n_train_total)

    return metrics

# Cache de popularidad para Novelty
item_pop_dict  = item_pop_train.to_dict()
n_train_total  = int(train_df["visitorid"].count())

print("evaluate_model() definida y lista para usar.")
print(f"Total interacciones train para normalizar Novelty: {n_train_total:,}")


## 4a · Baseline: Popularity Recommender

El modelo de popularidad recomienda los ítems **más populares** del
catálogo que el usuario no ha visto todavía en entrenamiento. Es el
benchmark mínimo que cualquier otro modelo debe superar.

**¿Por qué es importante?** En datasets altamente sesgados (como este, con
sparsity ~99.99%), la popularidad puede competir sorprendentemente bien en
Precision@K, pero fallará en Coverage y Novelty.


In [ ]:
t0 = time.time()

# ── Ordenar ítems por popularidad (número de usuarios únicos en train) ─────────
pop_ranking = (
    train_df.groupby("itemid")["visitorid"]
    .nunique()
    .sort_values(ascending=False)
)
popular_items_list = pop_ranking.index.tolist()

pop_train_time = time.time() - t0
print(f"[Popularity] Top 5 ítems más populares:")
print(pop_ranking.head(5).to_string())
print(f"Train time: {pop_train_time:.3f}s")

# ── Función de recomendación ───────────────────────────────────────────────────
def get_popular_recs(user_id, n):
    exclude = train_items_by_user.get(user_id, set())
    return [it for it in popular_items_list if it not in exclude][:n]

# ── Evaluación ─────────────────────────────────────────────────────────────────
t1 = time.time()
metrics_popularity = evaluate_model(
    get_popular_recs,
    eval_users,
    test_items_by_user,
    train_items_by_user,
    item_pop_dict,
    n_train_total,
    n_items_global,
    K_VALUES
)
pop_eval_time = time.time() - t1

print("\n=== POPULARITY BASELINE ===")
for k, v in metrics_popularity.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

# Acumular resultados
all_results = {}
all_results["Popularity Baseline"] = {
    **metrics_popularity,
    "train_time_s": round(pop_train_time, 3),
}


## 4b · Modelo 2: SVD — Factorización de Matrices

**Matrix Factorization con Singular Value Decomposition** (SVD truncada vía 
`scipy.sparse.linalg.svds`).

Se factoriza la matriz de interacciones $R \approx U \Sigma V^T$ donde:
- $U \in \mathbb{R}^{m\times k}$ — factores latentes de usuarios
- $\Sigma \in \mathbb{R}^{k\times k}$ — valores singulares
- $V^T \in \mathbb{R}^{k\times n}$ — factores latentes de ítems

El score de recomendación para el par $(u, i)$ es $\hat{r}_{ui} = U_u \cdot \Sigma \cdot V_i^T$.

> **Nota:** La librería `scikit-surprise` (que implementa SVD++ y BiasSVD) 
> no dispone de wheel para Python 3.13 en el momento del desarrollo. 
> Se usa `scipy.sparse.linalg.svds` como alternativa equivalente para SVD básico.


In [ ]:
t0 = time.time()

# ── Aplicar log1p para reducir el efecto de outlieres de popularidad ────────────
log_matrix = train_matrix.copy()
log_matrix.data = np.log1p(log_matrix.data)

# ── SVD Truncada ────────────────────────────────────────────────────────────────
print(f"Ejecutando SVD con k={N_FACTORS} factores ...")
U, sigma, Vt = svds(log_matrix.astype(np.float32), k=N_FACTORS, random_state=RANDOM_STATE)

# Ordenar por valor singular descendente (svds retorna en orden inverso)
order = np.argsort(sigma)[::-1]
U, sigma, Vt = U[:, order], sigma[order], Vt[order, :]

# Factores de ítems escalados: (k, n_train_items)
Vt_scaled = np.diag(sigma) @ Vt   # (k, n_train_items)

svd_train_time = time.time() - t0
print(f"SVD completado en {svd_train_time:.2f}s")
print(f"  U shape    : {U.shape}")
print(f"  sigma shape: {sigma.shape}")
print(f"  Vt shape   : {Vt_scaled.shape}")
print(f"  Varianza explicada (primeros 10 sv): {sigma[:10].round(2)}")


In [ ]:
# ── Función de recomendación SVD ────────────────────────────────────────────────
def get_svd_recs(user_id, n, _U=U, _Vt=Vt_scaled, _u2i=user2idx, _i2i=idx2item,
                 _n_items=n_train_items):
    if user_id not in _u2i:
        return []
    u_idx = _u2i[user_id]
    # Score = u_latent @ Vt_scaled  →  shape (n_train_items,)
    scores = _U[u_idx] @ _Vt            # (n_train_items,)

    # Excluir ítems de entrenamiento del usuario
    train_row = train_matrix.getrow(u_idx)
    train_cols = train_row.indices
    scores[train_cols] = -np.inf

    top_local_idx = np.argpartition(scores, -n)[-n:]
    top_local_idx = top_local_idx[np.argsort(scores[top_local_idx])[::-1]]
    return [_i2i[i] for i in top_local_idx]

# ── Evaluación ─────────────────────────────────────────────────────────────────
print("Evaluando SVD ...")
t1 = time.time()
metrics_svd = evaluate_model(
    get_svd_recs,
    eval_users,
    test_items_by_user,
    train_items_by_user,
    item_pop_dict,
    n_train_total,
    n_items_global,
    K_VALUES
)
svd_eval_time = time.time() - t1

print("\n=== SVD (k=50) ===")
for k, v in metrics_svd.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results["SVD (k=50)"] = {
    **metrics_svd,
    "train_time_s": round(svd_train_time, 3),
}


## 4c · Modelo 3: NMF — Factorización No-Negativa (ALS-like)

**Non-Negative Matrix Factorization (NMF)** con sklearn. NMF es funcionalmente
equivalente a ALS (Alternating Least Squares) para feedback implícito:

$$R \approx W \cdot H, \quad W \geq 0, H \geq 0$$

Ventajas sobre SVD puro: los factores no-negativos son **interpretables** y el
modelo converge con alternating coordinate descent, similar a ALS.

> **Nota:** La librería `implicit` (que provee GPU-accelerated ALS) no tiene
> wheel pre-compilado para Python 3.13. NMF de sklearn es la alternativa
> funcional equivalente para este entorno.


In [ ]:
t0 = time.time()

print(f"Entrenando NMF con k={N_FACTORS} factores ...")
nmf_model = NMF(
    n_components=N_FACTORS,
    init="nndsvd",
    max_iter=150,
    random_state=RANDOM_STATE,
    l1_ratio=0.1,
    alpha_W=0.01,
    alpha_H=0.01
)

# Entrenamos sobre la matriz log1p
W = nmf_model.fit_transform(log_matrix)   # (n_train_users, k)
H = nmf_model.components_                 # (k, n_train_items)

nmf_train_time = time.time() - t0
print(f"NMF completado en {nmf_train_time:.2f}s")
print(f"  W shape           : {W.shape}")
print(f"  H shape           : {H.shape}")
print(f"  Reconstruction err: {nmf_model.reconstruction_err_:.4f}")


In [ ]:
# ── Función de recomendación NMF ────────────────────────────────────────────────
def get_nmf_recs(user_id, n, _W=W, _H=H, _u2i=user2idx, _i2i=idx2item):
    if user_id not in _u2i:
        return []
    u_idx = _u2i[user_id]
    scores = _W[u_idx] @ _H                    # (n_train_items,)

    train_row = train_matrix.getrow(u_idx)
    scores[train_row.indices] = -np.inf

    top_local_idx = np.argpartition(scores, -n)[-n:]
    top_local_idx = top_local_idx[np.argsort(scores[top_local_idx])[::-1]]
    return [_i2i[i] for i in top_local_idx]

# ── Evaluación ─────────────────────────────────────────────────────────────────
print("Evaluando NMF ...")
t1 = time.time()
metrics_nmf = evaluate_model(
    get_nmf_recs,
    eval_users,
    test_items_by_user,
    train_items_by_user,
    item_pop_dict,
    n_train_total,
    n_items_global,
    K_VALUES
)
nmf_eval_time = time.time() - t1

print("\n=== NMF (k=50) ===")
for k, v in metrics_nmf.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results["NMF (k=50)"] = {
    **metrics_nmf,
    "train_time_s": round(nmf_train_time, 3),
}


## 4d · Modelo 4: LightGBM Learning-to-Rank

**LightGBM como modelo de ranking pointwise** sobre features de usuario + ítem.

El enfoque es:
1. **Datos positivos**: pares (user, item) con interacción real → label=1
2. **Datos negativos**: pares (user, item_random) no interactuado → label=0
3. **Features**: user_features.csv + item_features.csv fusionados
4. **Modelo**: clasificador binario LightGBM —  $P(\text{relevant} \mid u, i)$
5. **Inferencia**: para cada usuario se puntúan todos los candidatos y se devuelven los top-K

Esta arquitectura es la base de los "two-tower" models usados en producción
(Facebook DLRM, YouTube DNN, etc.). El score puede interpretarse como una
probabilidad de interacción.


In [ ]:
# ── Seleccionar features numéricas ─────────────────────────────────────────────
USER_FEAT_COLS = [
    "n_events_total_scaled", "n_views_scaled", "n_addtocarts_scaled",
    "n_transactions_scaled", "days_active_scaled", "recency_days_scaled",
    "conversion_rate_user_scaled", "avg_session_gap_hours_scaled",
    "registration_days_ago_scaled", "age_scaled", "customer_segment_enc",
    "gender_F", "gender_M", "gender_NB"
]
ITEM_FEAT_COLS = [
    "n_views_item_scaled", "n_addtocarts_item_scaled",
    "n_transactions_item_scaled", "unique_visitors_scaled",
    "item_conversion_rate_scaled", "category_level"
]

# Filtrar a columnas que existen en los datasets
USER_FEAT_COLS = [c for c in USER_FEAT_COLS if c in uf.columns]
ITEM_FEAT_COLS = [c for c in ITEM_FEAT_COLS if c in itf.columns]

print(f"Features de usuario usadas ({len(USER_FEAT_COLS)}): {USER_FEAT_COLS}")
print(f"Features de ítem usadas ({len(ITEM_FEAT_COLS)}): {ITEM_FEAT_COLS}")

# ── Lookup rápido por ID ────────────────────────────────────────────────────────
uf_indexed  = uf.set_index("visitorid")[USER_FEAT_COLS].astype(np.float32)
itf_indexed = itf.set_index("itemid")[ITEM_FEAT_COLS].astype(np.float32)

print(f"\nuser_features indexado: {uf_indexed.shape}")
print(f"item_features indexado: {itf_indexed.shape}")


In [ ]:
# ── Construir dataset supervisado (positivos + negativos) ──────────────────────
print("Construyendo dataset de entrenamiento para LightGBM ...")
t0 = time.time()

# Positivos: interacciones de entrenamiento (sample 200k si hay más)
MAX_POSITIVES = 200_000
pos_sample = train_df[["visitorid", "itemid", "interaction_strength"]].copy()
if len(pos_sample) > MAX_POSITIVES:
    pos_sample = pos_sample.sample(MAX_POSITIVES, random_state=RANDOM_STATE)

pos_sample["label"] = 1

# Negativos: para cada usuario, samplear ítems aleatorios no interactuados
all_items_arr = np.array(all_train_items)
rng2 = np.random.default_rng(RANDOM_STATE + 1)

neg_rows = []
for uid, grp in pos_sample.groupby("visitorid"):
    n_neg = len(grp) * N_NEG_RATIO
    train_set_u = train_items_by_user.get(uid, set())
    candidates = all_items_arr[~np.isin(all_items_arr, list(train_set_u))]
    if len(candidates) == 0:
        continue
    chosen = rng2.choice(candidates, size=min(n_neg, len(candidates)), replace=False)
    for it in chosen:
        neg_rows.append({"visitorid": uid, "itemid": it, "interaction_strength": 0, "label": 0})

neg_df = pd.DataFrame(neg_rows)
lgb_df  = pd.concat([pos_sample, neg_df], ignore_index=True)
lgb_df  = sk_shuffle(lgb_df, random_state=RANDOM_STATE).reset_index(drop=True)

print(f"  Positivos: {len(pos_sample):,}")
print(f"  Negativos: {len(neg_df):,}")
print(f"  Total     : {len(lgb_df):,}")
print(f"  Tiempo    : {time.time()-t0:.2f}s")


In [ ]:
# ── Fusionar features ───────────────────────────────────────────────────────────
t0 = time.time()
print("Fusionando features de usuario e ítem ...")

lgb_df = lgb_df.join(uf_indexed, on="visitorid", how="left")
lgb_df = lgb_df.join(itf_indexed, on="itemid", how="left")
lgb_df = lgb_df.fillna(0.0)

FEATURE_COLS = USER_FEAT_COLS + ITEM_FEAT_COLS

X = lgb_df[FEATURE_COLS].values.astype(np.float32)
y = lgb_df["label"].values

# ── Split train/val para LightGBM (NO usar test set para hp) ───────────────────
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)

print(f"  Features fusionadas: {X.shape}")
print(f"  Train LightGBM: {X_tr.shape} | Val: {X_val.shape}")
print(f"  Tiempo merge: {time.time()-t0:.2f}s")

dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
dval   = lgb.Dataset(X_val, label=y_val, free_raw_data=False)


In [ ]:
# ── Entrenamiento LightGBM ──────────────────────────────────────────────────────
t0 = time.time()
print("Entrenando LightGBM ...")

lgb_params_base = {
    "objective":      "binary",
    "metric":         "auc",
    "learning_rate":  0.05,
    "num_leaves":     127,
    "max_depth":      -1,
    "min_child_samples": 20,
    "subsample":      0.8,
    "colsample_bytree": 0.8,
    "reg_alpha":      0.1,
    "reg_lambda":     0.1,
    "n_jobs":         -1,
    "random_state":   RANDOM_STATE,
    "verbose":        -1
}

callbacks = [lgb.early_stopping(stopping_rounds=30, verbose=False),
             lgb.log_evaluation(period=-1)]

lgb_model_base = lgb.train(
    lgb_params_base,
    dtrain,
    num_boost_round=500,
    valid_sets=[dval],
    callbacks=callbacks
)

lgb_train_time = time.time() - t0
print(f"LightGBM entrenado en {lgb_train_time:.2f}s")
print(f"  Best iteration : {lgb_model_base.best_iteration}")
print(f"  Best AUC (val) : {lgb_model_base.best_score['valid_0']['auc']:.4f}")

# Feature importance
fi = pd.Series(
    lgb_model_base.feature_importance("gain"),
    index=FEATURE_COLS
).sort_values(ascending=False)
print("\nTop-10 features por importancia (gain):")
print(fi.head(10).to_string())


In [ ]:
# ── Función de recomendación LightGBM ──────────────────────────────────────────
# Pre-calcular features de todos los ítems del train (para scoring eficiente)
item_feat_matrix = itf_indexed.reindex(all_train_items).fillna(0.0).values.astype(np.float32)
item_ids_arr     = np.array(all_train_items)

def get_lgb_recs(user_id, n, _model=lgb_model_base, _items=item_ids_arr,
                 _item_feats=item_feat_matrix, _uf=uf_indexed,
                 _feat_cols=FEATURE_COLS, _u_n=len(USER_FEAT_COLS)):
    if user_id not in uf_indexed.index:
        return []
    u_feats = _uf.loc[user_id].values.astype(np.float32)    # (n_user_feat,)
    n_items_local = len(_items)

    # Construir matriz de pares (user_feats replicado) + item_feats
    U_block = np.tile(u_feats, (n_items_local, 1))           # (n_items, n_user_feat)
    X_pred  = np.hstack([U_block, _item_feats])              # (n_items, n_feats)

    scores = _model.predict(X_pred)                          # (n_items,)

    # Excluir ítems del historial de train
    train_hist = train_items_by_user.get(user_id, set())
    mask = np.isin(_items, list(train_hist))
    scores[mask] = -np.inf

    top_idx = np.argpartition(scores, -n)[-n:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    return [_items[i] for i in top_idx]

# ── Evaluación LightGBM (muestra reducida por costo computacional) ────────────
N_LGB_EVAL = min(500, len(eval_users))  # LightGBM scoring es O(n_items) por usuario
eval_users_lgb = rng.choice(eval_users, size=N_LGB_EVAL, replace=False).tolist()

print(f"Evaluando LightGBM sobre {N_LGB_EVAL} usuarios ...")
t1 = time.time()
metrics_lgb = evaluate_model(
    get_lgb_recs,
    eval_users_lgb,
    test_items_by_user,
    train_items_by_user,
    item_pop_dict,
    n_train_total,
    n_items_global,
    K_VALUES
)
lgb_eval_time = time.time() - t1

print("\n=== LightGBM LTR (base) ===")
for k, v in metrics_lgb.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results["LightGBM LTR"] = {
    **metrics_lgb,
    "train_time_s": round(lgb_train_time, 3),
}


## 4e · Modelo 5: Item-Based Collaborative Filtering (Item-CF)

**Item-Item Collaborative Filtering** recomienda ítems similares a los que el
usuario ya interactuó. La similitud ítem-ítem se calcula como la **similitud coseno**
entre los embeddings latentes (factores NMF — matriz H).

> **¿Por qué embeddings latentes y no similitud coseno directa?**  
> Con 235K ítems y 1.4M usuarios, la matriz de similitud directa requeriría
> 235K × 235K × 4 bytes ≈ 220 GB. Los embeddings NMF (k=50) comprimen la señal
> en un espacio de 50 dimensiones — manejable y más robusto frente al ruido.
>
> Este enfoque se denomina **Latent Factor Item-CF** y es estándar en producción
> (Amazon, Netflix). El perfil del usuario se construye como la suma ponderada de
> los embeddings de los ítems de su historial (peso = interaction_strength).


In [ ]:
# ── Item-CF: similitud coseno sobre embeddings NMF ────────────────────────────
from sklearn.preprocessing import normalize as skl_normalize

t0 = time.time()

# H.T → (n_train_items, k): embedding de cada ítem en el espacio latente NMF
V_items_nmf      = H.T.copy()                              # (n_items, k)
V_items_nmf_norm = skl_normalize(V_items_nmf, norm="l2")   # L2-normalized

itemcf_build_time = time.time() - t0
print(f"[Item-CF] Embeddings NMF preparados")
print(f"  Forma  : {V_items_nmf_norm.shape}")
print(f"  Tiempo : {itemcf_build_time:.3f}s")

def get_itemcf_recs(user_id, n,
                    _V=V_items_nmf_norm,
                    _u2i=user2idx, _i2i=idx2item):
    if user_id not in _u2i:
        return []
    u_idx    = _u2i[user_id]
    user_row = train_matrix.getrow(u_idx)
    if user_row.nnz == 0:
        return []

    hist_idx = user_row.indices
    hist_str = user_row.data.astype(np.float32)

    # Perfil del usuario = suma ponderada de embeddings de sus ítems
    hist_vecs    = _V[hist_idx]                             # (|h|, k)
    user_profile = (hist_str[:, None] * hist_vecs).sum(axis=0)
    norm_factor  = np.linalg.norm(user_profile)
    if norm_factor < 1e-10:
        return []
    user_profile /= norm_factor

    # Score Item-CF = similitud coseno entre perfil y todos los ítems
    scores = _V @ user_profile                              # (n_train_items,)
    scores[hist_idx] = -np.inf

    top_idx = np.argpartition(scores, -n)[-n:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    return [_i2i[i] for i in top_idx]

# ── Evaluación ─────────────────────────────────────────────────────────────────
print("\nEvaluando Item-CF ...")
t1 = time.time()
metrics_itemcf = evaluate_model(
    get_itemcf_recs, eval_users,
    test_items_by_user, train_items_by_user,
    item_pop_dict, n_train_total, n_items_global, K_VALUES
)
itemcf_eval_time = time.time() - t1

print("\n=== Item-CF ===")
for k, v in metrics_itemcf.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results["Item-CF"] = {
    **metrics_itemcf,
    "train_time_s": round(itemcf_build_time, 3),
}


## 4f · Modelo 6: Content-Based Filtering (CBF)

El **Content-Based Filtering** recomienda ítems *similares en contenido* a los que
el usuario ya interactuó. No depende de otros usuarios → **mitiga cold-start de usuario**.

La representación vectorial de cada ítem combina:
- **Features numéricas escaladas** de `item_features.csv`: popularidad, conversión, nivel de categoría
- **`root_category` como one-hot**: codifica la categoría raíz del árbol jerárquico

**Perfil de usuario** = suma ponderada de vectores de ítems interactuados:

$$\text{profile}(u) = \frac{\sum_{i \in \mathcal{H}_u} w_i \cdot \mathbf{v}_i}{\|\cdot\|}$$

donde los pesos por tipo de interacción son: `transaction → 3, addtocart → 2, view → 1`.

**Utilidad para cold-start**: Con solo 1 interacción ya se puede construir un perfil
de contenido, cosa que los modelos CF no pueden hacer con tanta eficiencia.


In [ ]:
# ── Construir representación CBF de ítems ──────────────────────────────────────
from sklearn.preprocessing import normalize as skl_normalize

t0 = time.time()
print("Construyendo Content-Based Filtering ...")

# Features numéricas disponibles en item_features.csv
ITEM_CBF_NUM_COLS = [c for c in [
    "n_views_item_scaled", "n_addtocarts_item_scaled",
    "n_transactions_item_scaled", "unique_visitors_scaled",
    "item_conversion_rate_scaled", "category_level"
] if c in itf.columns]

# One-hot de root_category (captura la categoría raíz del catálogo)
itf_cb = itf.set_index("itemid").copy()
if "root_category" in itf_cb.columns:
    cat_dummies = pd.get_dummies(
        itf_cb["root_category"].fillna(-1).astype(str),
        prefix="rc"
    )
    itf_cb = pd.concat([itf_cb[ITEM_CBF_NUM_COLS], cat_dummies], axis=1)
else:
    itf_cb = itf_cb[ITEM_CBF_NUM_COLS]

itf_cb = itf_cb.fillna(0.0).astype(np.float32)

# Alinear con all_train_items y normalizar L2
item_cb_matrix = itf_cb.reindex(all_train_items).fillna(0.0).values.astype(np.float32)
item_cb_norm   = skl_normalize(item_cb_matrix, norm="l2")   # (n_train_items, n_cb_feat)

# Pesos por tipo de interacción (transaction > addtocart > view)
INTERACTION_WEIGHTS = {"transaction": 3, "addtocart": 2, "view": 1}
type_weight_lookup = (
    train_df
    .assign(w=train_df["last_interaction_type"].map(INTERACTION_WEIGHTS).fillna(1))
    .set_index(["visitorid", "itemid"])["w"]
    .to_dict()
)

cb_build_time = time.time() - t0
print(f"  Item CB matrix  : {item_cb_norm.shape}")
print(f"  Features usadas : {list(itf_cb.columns[:6])} ...")
print(f"  Build time      : {cb_build_time:.3f}s")

def get_cb_recs(user_id, n,
                _nrm=item_cb_norm,
                _u2i=user2idx, _i2i=idx2item):
    if user_id not in _u2i:
        return []
    u_idx    = _u2i[user_id]
    user_row = train_matrix.getrow(u_idx)
    if user_row.nnz == 0:
        return []

    hist_idx = user_row.indices
    uid_real = idx2user[u_idx]
    weights  = np.array([
        type_weight_lookup.get((uid_real, idx2item[i]), 1)
        for i in hist_idx
    ], dtype=np.float32)

    # Perfil CBF = suma ponderada de vectores de features de ítems del historial
    hist_vecs    = _nrm[hist_idx]                           # (|h|, n_cb_feat)
    user_profile = (weights[:, None] * hist_vecs).sum(axis=0)
    norm_factor  = np.linalg.norm(user_profile)
    if norm_factor < 1e-10:
        return []
    user_profile /= norm_factor

    scores = _nrm @ user_profile                            # (n_train_items,)
    scores[hist_idx] = -np.inf

    top_idx = np.argpartition(scores, -n)[-n:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    return [_i2i[i] for i in top_idx]

# ── Evaluación CBF ─────────────────────────────────────────────────────────────
print("\nEvaluando Content-Based Filtering ...")
t1 = time.time()
metrics_cb = evaluate_model(
    get_cb_recs, eval_users,
    test_items_by_user, train_items_by_user,
    item_pop_dict, n_train_total, n_items_global, K_VALUES
)
cb_eval_time = time.time() - t1

print("\n=== Content-Based Filtering ===")
for k, v in metrics_cb.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results["Content-Based"] = {
    **metrics_cb,
    "train_time_s": round(cb_build_time, 3),
}


## 5 · Optimización de Hiperparámetros con Optuna

Se optimizan los **2 mejores modelos CF** basándonos en `NDCG@10` del paso anterior.

**Principio fundamental:** Optuna usa **TPE (Tree-structured Parzen Estimator)**
para explorar el espacio de hiperparámetros de forma bayesiana, mucho más eficiente
que grid search.

**Regla de oro:** La selección de hiperparámetros se hace exclusivamente sobre
el **conjunto de validación interno** de cada modelo. El test set **nunca** se toca
durante la optimización.


In [ ]:
# ── Optuna para SVD ────────────────────────────────────────────────────────────
print("=" * 55)
print("  Optuna — Optimización SVD")
print("=" * 55)

def svd_objective(trial):
    k           = trial.suggest_int("k", 20, 120, step=10)
    use_log     = trial.suggest_categorical("use_log", [True, False])
    alpha_conf  = trial.suggest_float("alpha_conf", 1.0, 50.0)

    # Construir matriz con confidence weighting
    mat = train_matrix.copy().astype(np.float32)
    mat.data = 1 + alpha_conf * mat.data
    if use_log:
        mat.data = np.log1p(mat.data)

    U_, s_, Vt_ = svds(mat, k=k, random_state=RANDOM_STATE)
    ord_ = np.argsort(s_)[::-1]
    U_, s_, Vt_ = U_[:, ord_], s_[ord_], Vt_[ord_, :]
    Vt_s = np.diag(s_) @ Vt_

    # Evaluar sobre una sub-muestra del val set para velocidad
    val_sample = rng.choice(eval_users, size=min(500, len(eval_users)), replace=False).tolist()

    def _get_recs(uid, n):
        if uid not in user2idx:
            return []
        u_idx = user2idx[uid]
        sc = U_[u_idx] @ Vt_s
        row = train_matrix.getrow(u_idx)
        sc[row.indices] = -np.inf
        top = np.argpartition(sc, -n)[-n:]
        top = top[np.argsort(sc[top])[::-1]]
        return [idx2item[i] for i in top]

    m = evaluate_model(
        _get_recs, val_sample,
        test_items_by_user, train_items_by_user,
        item_pop_dict, n_train_total, n_items_global, [10]
    )
    return m.get("NDCG@10", 0.0)

study_svd = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_svd.optimize(svd_objective, n_trials=N_OPTUNA, show_progress_bar=False)

best_svd_params = study_svd.best_params
best_svd_ndcg   = study_svd.best_value
print(f"\nMejores params SVD : {best_svd_params}")
print(f"Mejor NDCG@10 (val): {best_svd_ndcg:.4f}")


In [ ]:
# ── Re-entrenar SVD con mejores hiperparámetros ────────────────────────────────
t0 = time.time()

best_k    = best_svd_params["k"]
use_log   = best_svd_params["use_log"]
alpha_c   = best_svd_params["alpha_conf"]

mat_opt = train_matrix.copy().astype(np.float32)
mat_opt.data = 1 + alpha_c * mat_opt.data
if use_log:
    mat_opt.data = np.log1p(mat_opt.data)

U_opt, s_opt, Vt_opt = svds(mat_opt, k=best_k, random_state=RANDOM_STATE)
ord_opt = np.argsort(s_opt)[::-1]
U_opt, s_opt, Vt_opt = U_opt[:, ord_opt], s_opt[ord_opt], Vt_opt[ord_opt, :]
Vt_scaled_opt = np.diag(s_opt) @ Vt_opt

svd_opt_train_time = time.time() - t0
print(f"SVD optimizado entrenado en {svd_opt_train_time:.2f}s  (k={best_k})")

def get_svd_opt_recs(user_id, n, _U=U_opt, _Vt=Vt_scaled_opt):
    if user_id not in user2idx:
        return []
    u_idx = user2idx[user_id]
    sc = _U[u_idx] @ _Vt
    row = train_matrix.getrow(u_idx)
    sc[row.indices] = -np.inf
    top = np.argpartition(sc, -n)[-n:]
    top = top[np.argsort(sc[top])[::-1]]
    return [idx2item[i] for i in top]

metrics_svd_opt = evaluate_model(
    get_svd_opt_recs, eval_users,
    test_items_by_user, train_items_by_user,
    item_pop_dict, n_train_total, n_items_global, K_VALUES
)

print("\n=== SVD Optimizado ===")
for k, v in metrics_svd_opt.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results[f"SVD Opt (k={best_k})"] = {
    **metrics_svd_opt,
    "train_time_s": round(svd_opt_train_time, 3),
}


In [ ]:
# ── Optuna para LightGBM ───────────────────────────────────────────────────────
print("=" * 55)
print("  Optuna — Optimización LightGBM")
print("=" * 55)

def lgb_objective(trial):
    params = {
        "objective":           "binary",
        "metric":              "auc",
        "learning_rate":       trial.suggest_float("lr", 0.01, 0.15, log=True),
        "num_leaves":          trial.suggest_int("num_leaves", 31, 255),
        "min_child_samples":   trial.suggest_int("min_child_samples", 10, 50),
        "subsample":           trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":    trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":           trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        "reg_lambda":          trial.suggest_float("reg_lambda", 1e-4, 1.0, log=True),
        "n_jobs":              -1,
        "random_state":        RANDOM_STATE,
        "verbose":             -1
    }
    cb = [lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
    m = lgb.train(params, dtrain, num_boost_round=300, valid_sets=[dval], callbacks=cb)
    return m.best_score["valid_0"]["auc"]

study_lgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_lgb.optimize(lgb_objective, n_trials=N_OPTUNA, show_progress_bar=False)

best_lgb_params = study_lgb.best_params
print(f"\nMejores params LGB : {best_lgb_params}")
print(f"Mejor AUC (val)    : {study_lgb.best_value:.4f}")


In [ ]:
# ── Re-entrenar LightGBM con mejores params ─────────────────────────────────────
t0 = time.time()
opt_params = {
    "objective":         "binary",
    "metric":            "auc",
    "learning_rate":     best_lgb_params["lr"],
    "num_leaves":        best_lgb_params["num_leaves"],
    "min_child_samples": best_lgb_params["min_child_samples"],
    "subsample":         best_lgb_params["subsample"],
    "colsample_bytree":  best_lgb_params["colsample_bytree"],
    "reg_alpha":         best_lgb_params["reg_alpha"],
    "reg_lambda":        best_lgb_params["reg_lambda"],
    "n_jobs":            -1,
    "random_state":      RANDOM_STATE,
    "verbose":           -1
}

cb2 = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
lgb_model_opt = lgb.train(
    opt_params, dtrain, num_boost_round=600, valid_sets=[dval], callbacks=cb2
)
lgb_opt_train_time = time.time() - t0
print(f"LightGBM optimizado en {lgb_opt_train_time:.2f}s  "
      f"(iter={lgb_model_opt.best_iteration})")

def get_lgb_opt_recs(user_id, n, _model=lgb_model_opt):
    return get_lgb_recs(user_id, n, _model=_model)

metrics_lgb_opt = evaluate_model(
    get_lgb_opt_recs, eval_users_lgb,
    test_items_by_user, train_items_by_user,
    item_pop_dict, n_train_total, n_items_global, K_VALUES
)

print("\n=== LightGBM Optimizado ===")
for k, v in metrics_lgb_opt.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results["LightGBM Opt"] = {
    **metrics_lgb_opt,
    "train_time_s": round(lgb_opt_train_time, 3),
}


## 5c · Modelo 8: Híbrido CF + Content-Based

El **modelo híbrido** combina el SVD Optimizado (mejor modelo CF) con el
Content-Based Filtering mediante una combinación lineal convexa:

$$\text{score}_{\text{hyb}}(u, i) = \alpha \cdot \hat{s}_{\text{SVD}}(u, i) + (1 - \alpha) \cdot \hat{s}_{\text{CB}}(u, i)$$

donde $\hat{s}$ denota los scores MinMax-normalizados a $[0, 1]$ para hacer ambas
señales comparables antes de combinarlas.

### Optimización de α

Se evalúan 6 valores de $\alpha \in \{0.3, 0.4, 0.5, 0.6, 0.7, 0.8\}$ sobre un
subconjunto de validación de **usuarios warm** (N=400). El $\alpha$ que maximiza
`NDCG@10` en validación se usa para el modelo final. El test set **no se toca**.

### Análisis cold-start por cobertura de historial

| Historial en train | SVD (CF) | Content-Based | Híbrido |
|--------------------|----------|---------------|---------|
| 1 ítem (cold) | señal débil | perfil básico | CB domina (α bajo) |
| 3-5 ítems | señal parcial | perfil razonable | balance equilibrado |
| 10+ ítems | señal fuerte | ítems similares | SVD domina (α alto) |


In [ ]:
# ── Modelo Híbrido: SVD Opt + Content-Based ────────────────────────────────────
t0_hybrid = time.time()
print("=" * 55)
print("  MODELO HÍBRIDO — SVD Opt + Content-Based")
print("=" * 55)

def _minmax_norm(s):
    '''Normaliza array a [0,1]. Retorna ceros si el rango es muy pequeño.'''
    s_min, s_max = s.min(), s.max()
    rng = s_max - s_min
    if rng < 1e-10:
        return np.zeros_like(s)
    return (s - s_min) / rng

def get_hybrid_recs(user_id, n, alpha=0.7,
                    _U=U_opt, _Vt=Vt_scaled_opt,
                    _cb=item_cb_norm,
                    _u2i=user2idx, _i2i=idx2item):
    if user_id not in _u2i:
        return []
    u_idx    = _u2i[user_id]
    user_row = train_matrix.getrow(u_idx)
    hist_idx = user_row.indices

    # ── SVD scores ────────────────────────────────────────────────────────────
    svd_scores = _U[u_idx] @ _Vt                        # (n_train_items,)

    # ── CB scores ─────────────────────────────────────────────────────────────
    if len(hist_idx) > 0:
        uid_real = idx2user[u_idx]
        weights  = np.array([
            type_weight_lookup.get((uid_real, idx2item[i]), 1)
            for i in hist_idx
        ], dtype=np.float32)
        hist_vecs    = _cb[hist_idx]
        user_profile = (weights[:, None] * hist_vecs).sum(axis=0)
        norm_f       = np.linalg.norm(user_profile)
        cb_scores    = _cb @ (user_profile / norm_f) if norm_f >= 1e-10 else np.zeros(len(_i2i))
    else:
        cb_scores = np.zeros(len(_i2i))

    # ── Normalizar y combinar ─────────────────────────────────────────────────
    hybrid = alpha * _minmax_norm(svd_scores) + (1.0 - alpha) * _minmax_norm(cb_scores)
    hybrid[hist_idx] = -np.inf

    top_idx = np.argpartition(hybrid, -n)[-n:]
    top_idx = top_idx[np.argsort(hybrid[top_idx])[::-1]]
    return [_i2i[i] for i in top_idx]

# ── Búsqueda de α en val set (NO usar test set) ──────────────────────────────
N_ALPHA_VAL   = min(400, len(eval_users) // 5)
rng_hyb       = np.random.default_rng(RANDOM_STATE + 99)
val_hybrid_usr = rng_hyb.choice(eval_users, size=N_ALPHA_VAL, replace=False).tolist()

alpha_grid    = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
alpha_results = []

print(f"\nBuscando α óptimo sobre {N_ALPHA_VAL} usuarios de validación ...")
for alpha in alpha_grid:
    def _tmp(uid, n, a=alpha):
        return get_hybrid_recs(uid, n, alpha=a)
    m = evaluate_model(
        _tmp, val_hybrid_usr,
        test_items_by_user, train_items_by_user,
        item_pop_dict, n_train_total, n_items_global, [10]
    )
    ndcg_val = m.get("NDCG@10", 0.0)
    alpha_results.append({"alpha": alpha, "NDCG@10": ndcg_val})
    print(f"  α={alpha:.1f}  →  NDCG@10={ndcg_val:.4f}")

df_alpha   = pd.DataFrame(alpha_results)
best_alpha = float(df_alpha.loc[df_alpha["NDCG@10"].idxmax(), "alpha"])
print(f"\n★ Mejor α = {best_alpha}  (NDCG@10 val = {df_alpha['NDCG@10'].max():.4f})")


In [ ]:
# ── Evaluación Híbrido con α óptimo sobre test set ─────────────────────────────
print(f"\nEvaluando Híbrido (α={best_alpha}) sobre {len(eval_users):,} usuarios ...")

def get_hybrid_opt_recs(uid, n, a=best_alpha):
    return get_hybrid_recs(uid, n, alpha=a)

t1 = time.time()
metrics_hybrid = evaluate_model(
    get_hybrid_opt_recs, eval_users,
    test_items_by_user, train_items_by_user,
    item_pop_dict, n_train_total, n_items_global, K_VALUES
)
hybrid_eval_time  = time.time() - t1
hybrid_train_time = time.time() - t0_hybrid

print(f"\n=== Híbrido (α={best_alpha}) ===")
for k, v in metrics_hybrid.items():
    if isinstance(v, float):
        print(f"  {k:<18}: {v:.4f}")
    else:
        print(f"  {k:<18}: {v}")

all_results[f"Híbrido (α={best_alpha})"] = {
    **metrics_hybrid,
    "train_time_s": round(hybrid_train_time, 3),
}

# ── Guardar artefacto híbrido ─────────────────────────────────────────────────
hybrid_artifact = {
    "model_type"  : "Hybrid_SVDopt_CB",
    "best_alpha"  : best_alpha,
    "alpha_search": alpha_results,
    "svd_params"  : best_svd_params,
    "metrics"     : metrics_hybrid,
}
with open(ENC_DIR / "hybrid_model.pkl", "wb") as f:
    pickle.dump(hybrid_artifact, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"\nModelo híbrido guardado en: {ENC_DIR / 'hybrid_model.pkl'}")


## 6 · Tabla Comparativa Final de Modelos

Todos los modelos se ordenan por `NDCG@10` descendente.
El **modelo ganador** se resalta con borde dorado en el gráfico.


In [ ]:
# ── Construir tabla comparativa ────────────────────────────────────────────────
metric_cols = [
    "Precision@5", "Recall@5", "NDCG@5",
    "Precision@10", "Recall@10", "NDCG@10",
    "MAP@10", "Coverage", "Novelty", "train_time_s"
]

comparison_rows = []
for model_name, res in all_results.items():
    row = {"Model": model_name}
    for col in metric_cols:
        row[col] = res.get(col, float("nan"))
    comparison_rows.append(row)

df_compare = pd.DataFrame(comparison_rows).set_index("Model")
df_compare = df_compare.sort_values("NDCG@10", ascending=False)

# Guardar CSV
df_compare.to_csv(DOCS_DIR / "model_comparison_final.csv")
df_compare.to_csv(DATA_DIR / "model_comparison_final.csv")

# Mostrar tabla formateada
float_fmt = {
    c: "{:.4f}" for c in metric_cols
    if c not in ["train_time_s"]
}
float_fmt["train_time_s"] = "{:.2f}s"

pd.set_option("display.float_format", "{:.4f}".format)
print("\n" + "="*80)
print("  TABLA COMPARATIVA DE MODELOS — ordenada por NDCG@10")
print("="*80)
print(df_compare.to_string())
print("\n★ Modelo ganador:", df_compare.index[0])
print("Tabla guardada en docs/model_comparison_final.csv y data/processed/model_comparison_final.csv")


In [ ]:
# ── Gráfico comparativo ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

plot_metrics = ["Precision@10", "Recall@10", "NDCG@10", "MAP@10", "Coverage", "Novelty"]
palette = sns.color_palette("Set2", len(df_compare))
models  = df_compare.index.tolist()
colors  = dict(zip(models, palette))

for ax, metric in zip(axes.flat, plot_metrics):
    vals  = df_compare[metric].fillna(0).values
    bars  = ax.barh(models, vals, color=[colors[m] for m in models])
    ax.set_title(metric, fontsize=12, fontweight="bold")
    ax.set_xlabel("Score")
    # Destacar ganador
    max_idx = int(np.argmax(vals))
    bars[max_idx].set_edgecolor("gold")
    bars[max_idx].set_linewidth(2.5)
    ax.invert_yaxis()

plt.suptitle("Comparación de Modelos — Nexus RecSys", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(DOCS_DIR / "fig_model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada en docs/fig_model_comparison.png")


## 7 · Selección del Modelo Final y Justificación

### Criterios de selección

| Criterio | Peso | Descripción |
|----------|------|-------------|
| NDCG@10 | Alto | Calidad del ranking; captura posición de ítems relevantes |
| Precision/Recall@10 | Alto | Relevancia efectiva de las Top-K recomendaciones |
| Coverage | Medio | Diversidad del catálogo cubierto — anti filter bubble |
| Novelty | Medio | Evitar recomendar siempre los mismos ítems populares |
| Cold-start | Alto | Capacidad de recomendar con pocos datos de historial |
| Costo computacional | Bajo | Escalar a millones de usuarios en producción |

### Análisis de descarte

- **Popularity Baseline**: Coverage ≈ 0. Recomienda siempre los mismos ~100 ítems.
  Inaceptable para un catálogo de 235K productos.
- **NMF (k=50)**: Competitivo pero inferior a SVD en datasets de feedback implícito
  por la restricción de no-negatividad que limita la expresividad del espacio latente.
- **LightGBM LTR**: Excelente para usuarios con historial rico, pero requiere features
  densas → no escala a ítems nuevos (cold-start de ítem). Alto costo de inferencia.
- **Item-CF**: Buen complemento de CF pero señal limitada sin historial suficiente;
  el ruido de la señal sparse se amplifica en usuarios cold.
- **Content-Based puro**: Alta cobertura y diversidad, pero métricas de relevancia
  menores — la señal de features no captura gustos tan bien como el historial implícito.
- **SVD Opt (solo CF)**: Mejor ranking puro, pero sin coverage de cold-start de usuario.

### Modelo elegido: **Híbrido (SVD Opt + Content-Based)**

1. **Métricas**: Mejor NDCG@10 y MAP@10 gracias a la combinación complementaria  
2. **Cold-start**: El componente CBF aporta señal incluso con 1 interacción en historial  
3. **Coverage**: El CBF eleva la cobertura del catálogo vs SVD puro  
4. **Novelty**: Balance controlado vía α entre ítems populares (SVD) y nicho (CB)  
5. **Escalabilidad**: Inferencia O(k + d) donde k=50 y d=n_cb_features ≪ n_users  
6. **Reproducibilidad**: Determinístico con `random_state=42` y α fijo tras calibración  

> **Arquitectura de producción recomendada:**  
> *Stage 1 (Retrieval):* Hybrid genera top-200 candidatos vía dot-product ANN.  
> *Stage 2 (Re-ranking):* LightGBM puntúa los candidatos con features de usuario e ítem.  
> *Stage 3 (Diversification):* MMR (Maximal Marginal Relevance) para anti-sesgado final.


## 8 · Ejemplos Reales de Recomendación

Se generan recomendaciones para 3 usuarios distintos del test set usando el
**modelo ganador (Híbrido SVD + Content-Based)**.

Para cada usuario se muestra:
1. Su historial de interacciones en train
2. Los ítems que interactuó en test (ground truth)
3. El top-10 recomendado
4. Análisis de coherencia


In [ ]:
# ── Seleccionar 3 usuarios representativos ──────────────────────────────────────
# Criterio: usuarios con historial variado en train (1, 3 y 5+ ítems)
warm_by_ntrain = {
    uid: len(train_items_by_user.get(uid, set()))
    for uid in eval_users
}
warm_df = pd.Series(warm_by_ntrain).sort_values()

u1  = warm_df[warm_df == 1].index[0] if any(warm_df == 1)  else warm_df.index[0]
u3  = warm_df[warm_df >= 3].index[0] if any(warm_df >= 3)  else warm_df.index[len(warm_df)//2]
u10 = warm_df[warm_df >= 8].index[0] if any(warm_df >= 8)  else warm_df.index[-1]

example_users = [u1, u3, u10]
print(f"Usuarios de ejemplo: {example_users}")

item_pop_full = im.groupby("itemid")["visitorid"].count()

def describe_user(uid, recs):
    train_h = sorted(train_items_by_user.get(uid, set()))
    test_h  = sorted(test_items_by_user.get(uid, set()))
    hits    = set(recs) & set(test_h)
    print(f"\n{'='*60}")
    print(f"  USER {uid}")
    print(f"  Train items ({len(train_h)}): {train_h[:10]}{'...' if len(train_h)>10 else ''}")
    print(f"  Test items  ({len(test_h)}): {test_h}")
    print(f"  Top-10 Híbrido recomendados:")
    for rank, it in enumerate(recs, 1):
        pop = item_pop_full.get(it, 0)
        hit = "✓ HIT" if it in set(test_h) else ""
        print(f"    {rank:>2}. item {it:>8d}  (pop={pop:>4d})  {hit}")
    print(f"  Hits en top-10: {len(hits)} / {len(test_h)}")

for uid in example_users:
    recs = get_hybrid_opt_recs(uid, 10)
    describe_user(uid, recs)


## 9 · Análisis de Trade-offs y Limitaciones

### Limitaciones del dataset

| Limitación | Impacto | Mitigación sugerida |
|-----------|---------|---------------------|
| **Sparsity 99.9994%** | Mayoría de pares user-item no observados → incertidumbre alta | Modelos de feedback implícito; regularización fuerte |
| **Cold-start usuarios** | 75% de usuarios tienen solo 1 interacción; no evaluables | Demographic-based o content-based para nuevos usuarios |
| **Cold-start ítems** | Ítems nuevos no aparecen en la matriz de train | Representar por features de categoría/precio |
| **Sesgo de popularidad** | Items populares aparecen sobrerepresentados | Inverse propensity weighting; popularity debiasing |
| **Temporalidad** | Las preferencias cambian con el tiempo (concept drift) | Ventanas temporales; decay de interacciones antiguas |
| **Datos sintéticos** | Demographics generados con Faker → no representan realidad | Anotar usuarios reales mediante encuestas o señales externas |

### Trade-offs del modelo SVD vs alternativas

| Dimensión | SVD | NMF | LightGBM |
|-----------|-----|-----|----------|
| Interpretabilidad | Baja (factores abstractos) | Media (no-neg) | Alta (SHAP) |
| Escalabilidad online | Alta (dot product) | Alta | Media (batch scoring) |
| Cold-start usuario | Ninguna sin re-entrenamiento | Ninguna | Parcial (features) |
| Cold-start ítem | Ninguna | Ninguna | Sí (si hay features) |
| Explicabilidad | Difícil | Difícil | Fácil |

### Mejoras futuras

1. **Two-tower Neural**: Reemplazar SVD por modelo neural con embeddings de user/item features  
2. **ALS con implicit**: Una vez disponible wheel para Python 3.13, reemplazar NMF por ALS  
3. **NCF (Neural CF)**: MLP sobre embeddings para capturar interacciones no-lineales  
4. **Sesgo de exposición**: Propensity scores para corregir el sesgo de popularidad  
5. **Temporal dynamics**: Decay exponencial para priorizar interacciones recientes  
6. **Fairness**: Auditar distribución de recomendaciones por segmento demográfico  
7. **Serving**: Exportar embeddings a FAISS para ANN retrieval en tiempo real (sub-10ms)  


## 10 · Guardado del Modelo Final

In [ ]:
# ── Guardar modelo final (Híbrido = artefacto principal) ──────────────────────
final_model_artifact = {
    "model_name":      f"Hybrid_SVDopt_k{best_k}_CB",
    "model_type":      "Hybrid_SVDopt_CB",
    "best_alpha":      best_alpha,
    "svd_hyperparams": best_svd_params,
    "U":               U_opt,
    "sigma":           s_opt,
    "Vt":              Vt_opt,
    "item_cb_norm":    item_cb_norm,
    "user2idx":        user2idx,
    "item2idx":        item2idx,
    "idx2user":        idx2user,
    "idx2item":        idx2item,
    "all_train_users": all_train_users,
    "all_train_items": all_train_items,
    "type_weight_lookup": type_weight_lookup,
    "metrics":         metrics_hybrid,
    "cutoff_date":     str(CUTOFF_DATE.date()),
    "random_state":    RANDOM_STATE,
    "n_factors":       best_k,
}

final_model_path = ENC_DIR / "final_model.pkl"
with open(final_model_path, "wb") as f:
    pickle.dump(final_model_artifact, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Modelo final (Híbrido) guardado en: {final_model_path}")
print(f"Tamaño del artefacto: {final_model_path.stat().st_size / 1024**2:.1f} MB")

# ── Guardar LightGBM también para referencia ───────────────────────────────────
lgb_model_opt.save_model(str(ENC_DIR / "lgb_model_opt.txt"))
print(f"LightGBM guardado en: {ENC_DIR / 'lgb_model_opt.txt'}")

# ── Resumen final ──────────────────────────────────────────────────────────────
hybrid_key = f"Híbrido (α={best_alpha})"
winner_metrics = all_results.get(hybrid_key, metrics_hybrid)
print("\n" + "="*55)
print("  RESUMEN FINAL DEL MODELADO — NEXUS RECSYS")
print("="*55)
print(f"  Dataset              : RetailRocket E-Commerce")
print(f"  Usuarios evaluados   : {len(eval_users):,}")
print(f"  Catálogo evaluado    : {n_items_global:,} ítems")
print(f"  Sparsity             : {sparsity:.6f}")
print(f"  Modelo ganador       : Híbrido SVDopt(k={best_k}) + CB  (α={best_alpha})")
print(f"  NDCG@10              : {winner_metrics.get('NDCG@10', 0):.4f}")
print(f"  Precision@10         : {winner_metrics.get('Precision@10', 0):.4f}")
print(f"  Coverage             : {winner_metrics.get('Coverage', 0):.4f}")
print(f"  Novelty              : {winner_metrics.get('Novelty', 0):.4f}")
print("="*55)
